In [1]:
import ee   
#ee.Authenticate()

ee.Initialize(project="ee-bghosn4")

An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


c:\Users\admin\anaconda3\envs\arcpy_clone\lib\site-packages\google\api_core\_python_version_support.py:252: FutureWarning: You are using a Python version (3.9.11) past its end of life. Google will update google.api_core with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [2]:
dem = ee.Image('USGS/3DEP/10m')

geom = ee.Geometry.Point([-91.0989573,30.3529013])

geom_col = ee.FeatureCollection([geom])

elev = dem.sampleRegions(geom_col).getInfo()
elev['features'][0]['properties']['elevation']

c:\Users\admin\anaconda3\envs\arcpy_clone\lib\site-packages\ee\deprecation.py:214: DeprecationWarning: 

Attention required for USGS/3DEP/10m! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by USGS/3DEP/10m_collection

Learn more: https://developers.google.com/earth-engine/datasets/catalog/USGS_3DEP_10m

  warnings.warn(warning, category=DeprecationWarning)


3.220296621322632

In [3]:
import pandas as pd
table = pd.read_csv(r"C:\Users\admin\Desktop\Project 2\boundary.csv")
table.head()

,col,row,X,Y
0,4871,174,699102.887792,186780.445813
1,4871,174,699102.887792,186780.445813
2,4872,174,699105.887419,186780.445813
3,4870,175,699099.888166,186777.446186
4,4873,174,699108.887046,186780.445813


In [4]:
import arcpy
ra1 = arcpy.Raster(r"C:\Users\admin\Desktop\Project 2\flood_2class.tif")
print(ra1.spatialReference.factoryCode)

32119


In [5]:
import geopandas
gdf = geopandas.GeoDataFrame(table)

gdf.set_geometry( geopandas.points_from_xy(gdf['X'], gdf['Y']), inplace=True, crs=f'EPSG:{ra1.spatialReference.factoryCode}')

In [6]:
gdf.to_file(r"C:\Users\admin\Desktop\Project 2\boundary.shp")


In [7]:
shapefile = r"C:\Users\admin\Desktop\Project 2\boundary.shp"
arcpy.management.AddField(shapefile,'elevation',field_type='FLOAT')

<Result 'C:\\Users\\admin\\Desktop\\Project 2\\boundary.shp'>

In [8]:
geom_list = []
with arcpy.da.SearchCursor(shapefile,['SHAPE@XY'],spatial_reference = arcpy.SpatialReference(4326)
) as cursor:
    for row in cursor:
        X,Y = row[0]
        geom = ee.Geometry.Point([X,Y])
        geom_list.append(geom)
geom_col = ee.FeatureCollection(geom_list)
elev = dem.sampleRegions(geom_col).getInfo().get('features')

In [9]:
i = 0
with arcpy.da.UpdateCursor(shapefile,['elevation']) as cursor:
    for row in cursor:
        elevation = elev[i]['properties']['elevation']
        row[0] = elevation
        cursor.updateRow(row)
        i += 1